In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [ ]:
from yahooquery import Ticker
import pandas as pd

ticker = 'TLKM.JK'
start_date = '2024-12-13'
end_date = '2025-06-12'

t = Ticker(ticker)
data = t.history(start=start_date, end=end_date, interval='1d')

data = data.reset_index()

data = data[data['symbol'] == ticker]
print(data.head())

In [ ]:
telkom_data = pd.read_csv("/content/data_tlkm.csv", index_col=0)

In [ ]:
telkom_data.head(10)

,date,open,high,low,close,volume
0,2011-01-03,1362.50,1400.0,1325.00,1362.50,12152000
1,2011-01-04,1387.50,1400.0,1375.00,1387.50,4410000
2,2011-01-05,1412.50,1412.5,1362.50,1412.50,14656000
3,2011-01-06,1400.00,1412.5,1337.50,1400.00,6712000
4,2011-01-07,1337.50,1412.5,1300.00,1337.50,7298000
5,2011-01-10,1243.75,1275.0,1206.25,1243.75,33662000
6,2011-01-11,1250.00,1300.0,1250.00,1250.00,9214000
7,2011-01-12,1325.00,1325.0,1275.00,1325.00,6504000
8,2011-01-13,1300.00,1350.0,1275.00,1300.00,9120000
9,2011-01-14,1275.00,1300.0,1262.50,1275.00,12248000


In [ ]:
# telkom_data.drop(columns=['symbol'], inplace=True)
# telkom_data.reset_index(drop=True, inplace=True)
# telkom_data.drop(columns=['adjclose'], inplace=True)

KeyError: "['symbol'] not found in axis"

In [ ]:
telkom_data.head(6)

,date,open,high,low,close,volume
0,2011-01-03,1362.50,1400.0,1325.00,1362.50,12152000
1,2011-01-04,1387.50,1400.0,1375.00,1387.50,4410000
2,2011-01-05,1412.50,1412.5,1362.50,1412.50,14656000
3,2011-01-06,1400.00,1412.5,1337.50,1400.00,6712000
4,2011-01-07,1337.50,1412.5,1300.00,1337.50,7298000
5,2011-01-10,1243.75,1275.0,1206.25,1243.75,33662000


Assesing Data

In [ ]:
telkom_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3456 entries, 0 to 3455
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    3456 non-null   object 
 1   open    3456 non-null   float64
 2   high    3456 non-null   float64
 3   low     3456 non-null   float64
 4   close   3456 non-null   float64
 5   volume  3456 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 318.0+ KB


In [ ]:
#Cek Data Missing
telkom_data.isnull().sum()

,0
date,0
open,0
high,0
low,0
close,0
volume,0


In [ ]:
#Cek Duplikasi Data
telkom_data.duplicated().sum()

np.int64(0)

In [ ]:
#Mencari data inaccurate
telkom_data.describe()

,open,high,low,close,volume
count,3456.000000,3456.000000,3456.000000,3456.000000,3.456000e+03
mean,1360.342882,1382.983941,1336.433015,1359.307726,1.464238e+07
std,546.652446,554.429673,536.286888,546.191505,3.035011e+07
min,305.000000,332.500000,297.500000,300.000000,0.000000e+00
25%,976.250000,993.750000,971.250000,978.437500,1.874500e+06
50%,1343.750000,1362.500000,1318.750000,1343.750000,7.008600e+06
75%,1618.750000,1643.750000,1593.750000,1618.750000,1.644330e+07
max,3012.500000,3062.500000,2962.500000,3012.500000,5.864944e+08


In [ ]:
#cek Format Data
telkom_data.dtypes

,0
date,object
open,float64
high,float64
low,float64
close,float64
volume,int64


Cleaning Data

In [ ]:
#menampilkan nilai 0 pada volume
telkom_data[telkom_data['volume'] == 0]

,date,open,high,low,close,volume
124,2011-07-04,1262.50,1262.50,1262.50,1262.50,0
226,2011-12-01,1337.50,1337.50,1337.50,1337.50,0
311,2012-04-03,1287.50,1287.50,1287.50,1287.50,0
507,2013-01-22,1687.50,1687.50,1687.50,1687.50,0
569,2013-04-23,1562.50,1562.50,1562.50,1562.50,0
...,...,...,...,...,...,...
2296,2020-03-13,428.75,428.75,428.75,428.75,0
2484,2020-12-28,1412.50,1412.50,1412.50,1412.50,0
2491,2021-01-08,1475.00,1475.00,1475.00,1475.00,0
2492,2021-01-11,1475.00,1475.00,1475.00,1475.00,0


In [ ]:
telkom_data.shape

(3456, 6)

In [ ]:
#ubah 0 menjadi NaN
telkom_data['volume'] = telkom_data['volume'].replace(0, np.nan)
#interpolasi volume
telkom_data['volume'] = telkom_data['volume'].interpolate()

In [ ]:
telkom_data[telkom_data['volume'] == 0]

,date,open,high,low,close,volume
124,2011-07-04,1262.50,1262.50,1262.50,1262.50,0
226,2011-12-01,1337.50,1337.50,1337.50,1337.50,0
311,2012-04-03,1287.50,1287.50,1287.50,1287.50,0
507,2013-01-22,1687.50,1687.50,1687.50,1687.50,0
569,2013-04-23,1562.50,1562.50,1562.50,1562.50,0
...,...,...,...,...,...,...
2296,2020-03-13,428.75,428.75,428.75,428.75,0
2484,2020-12-28,1412.50,1412.50,1412.50,1412.50,0
2491,2021-01-08,1475.00,1475.00,1475.00,1475.00,0
2492,2021-01-11,1475.00,1475.00,1475.00,1475.00,0


In [ ]:
telkom_data[telkom_data['open'] == 0]

,date,open,high,low,close,volume


In [ ]:
telkom_data[telkom_data['high'] == 0]

,date,open,high,low,close,volume


In [ ]:
telkom_data[telkom_data['low'] == 0]

,date,open,high,low,close,volume


In [ ]:
telkom_data[telkom_data['close'] == 0]

,date,open,high,low,close,volume


In [ ]:
telkom_data.describe()

,open,high,low,close,volume
count,3456.000000,3456.000000,3456.000000,3456.000000,3.456000e+03
mean,1360.342882,1382.983941,1336.433015,1359.307726,1.464238e+07
std,546.652446,554.429673,536.286888,546.191505,3.035011e+07
min,305.000000,332.500000,297.500000,300.000000,0.000000e+00
25%,976.250000,993.750000,971.250000,978.437500,1.874500e+06
50%,1343.750000,1362.500000,1318.750000,1343.750000,7.008600e+06
75%,1618.750000,1643.750000,1593.750000,1618.750000,1.644330e+07
max,3012.500000,3062.500000,2962.500000,3012.500000,5.864944e+08


In [ ]:
#ubah date ke datetime
telkom_data['date'] = pd.to_datetime(telkom_data['date'])

In [ ]:
telkom_data.dtypes

,0
date,datetime64[ns]
open,float64
high,float64
low,float64
close,float64
volume,int64


In [ ]:
telkom_data.head()

,date,open,high,low,close,volume
0,2011-01-03,1362.5,1400.0,1325.0,1362.5,12152000
1,2011-01-04,1387.5,1400.0,1375.0,1387.5,4410000
2,2011-01-05,1412.5,1412.5,1362.5,1412.5,14656000
3,2011-01-06,1400.0,1412.5,1337.5,1400.0,6712000
4,2011-01-07,1337.5,1412.5,1300.0,1337.5,7298000


In [ ]:
telkom_data.shape

(3456, 6)

In [ ]:
telkom_data.head()

,date,open,high,low,close,volume
0,2011-01-03,1362.5,1400.0,1325.0,1362.5,12152000
1,2011-01-04,1387.5,1400.0,1375.0,1387.5,4410000
2,2011-01-05,1412.5,1412.5,1362.5,1412.5,14656000
3,2011-01-06,1400.0,1412.5,1337.5,1400.0,6712000
4,2011-01-07,1337.5,1412.5,1300.0,1337.5,7298000


In [ ]:
tahun_mulai = 2023
tahun_akhir = 2024
telkom_data = telkom_data[(telkom_data['date'].dt.year >= tahun_mulai) &
                          (telkom_data['date'].dt.year <= tahun_akhir)]

In [ ]:
telkom_data.shape

(3456, 6)

In [ ]:
#Mengubah kolom date menjadi indeks
telkom_data.set_index('date', inplace=True)

In [ ]:
telkom_data.shape

(3456, 5)

In [ ]:
# Plot stock data
plt.figure(figsize=(25, 8))
plt.title('Stock Prices History')
plt.plot(telkom_data['close'])
plt.xlabel('Date')
plt.ylabel('Prices')
plt.show()



In [ ]:
import matplotlib.pyplot as plt

plt.hist(telkom_data['close'], bins=30, edgecolor='black')
plt.xlabel('Harga Penutupan (Close)')
plt.ylabel('Frekuensi')
plt.title('Distribusi Harga Penutupan Saham')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.hist(telkom_data['open'], bins=30, edgecolor='black')
plt.xlabel('Harga Opem')
plt.ylabel('Frekuensi')
plt.title('Distribusi Harga Pembukaan Saham')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.hist(telkom_data['low'], bins=30, edgecolor='black')
plt.xlabel('Harga Low')
plt.ylabel('Frekuensi')
plt.title('Distribusi Harga Low Saham')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.hist(telkom_data['high'], bins=30, edgecolor='black')
plt.xlabel('Harga high')
plt.ylabel('Frekuensi')
plt.title('Distribusi Harga High Saham')
plt.show()


In [ ]:
telkom_data['close'].skew()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(telkom_data['volume'], bins=30, edgecolor='black')
plt.xlabel('volume')
plt.ylabel('Frekuensi')
plt.title('volume')
plt.show()


In [ ]:
telkom_data['volume'].skew()

In [ ]:
telkom_data.to_csv("data_TLKM_data_pred.csv")